# Views.py hice esto clase

In [ ]:
from django.http import HttpResponseRedirect

from django.contrib.auth.mixins import LoginRequiredMixin #<---- esta parte agrege para
#hacer la inportacion de LoginRequiredMixin para poner en las class necesarias

from django.views.generic import (
    ListView, 
    DetailView, 
    RedirectView,
    CreateView,
    UpdateView, 
    DeleteView, 
)
from django.shortcuts import render, get_object_or_404  

from .forms import ProductModelForm
from .mixins import TemplateTitleMixin
from .models import Product, DigitalProduct

class ProtectedListView(LoginRequiredMixin, ListView): # cree esta clase en la views.py
    model = Product
    template_name = "products/my-products.html"
    context_object_name = "products"

    def get_queryset(self): # este metodo sobre escribe el get_queryset para que funcione acorde a lo solicitado
        return Product.objects.filter(user=self.request.user)

class ProtectedProductUpdateView(LoginRequiredMixin, UpdateView):
    form_class = ProductModelForm
    template_name = "forms.html"

    def get_queryset(self):
        return Product.objects.filter(user=self.request.user)

    def get_success_url(self):
        return self.object.get_edit_url()
        
class ProtectedProductDeleteView(LoginRequiredMixin, DeleteView):
    template_name = "forms-delete.html"

    def get_queryset(self):
        return Product.objects.filter(user=self.request.user)

    def get_success_url(self):
        return "/products/products"

class ProtectedProductCreateView(LoginRequiredMixin, CreateView):
    form_class = ProductModelForm
    template_name = "forms.html"
    
    def form_valid(self, form):
        form.instance.user = self.request.user
        return super().form_valid(form)
    
    def form_invalid(self, form):
        return super().form_invalid(form)
    
class ProductIDRedirectView(RedirectView):
    def get_redirect_urls(self, *args, **kwargs):
        url_params = self.kwargs
        pk = url_params.get("pk")
        obj = get_object_or_404(Product, pk=pk)
        slug = obj.slug
        return f"/products/products/{slug}"
    
class ProductRedirectView(RedirectView):
    def get_redirect_urls(self, *args, **kwargs):
        url_params = self.kwargs
        slug = url_params.get("slug")
        return f"/products/products/{slug}"

class DigitalProductListView(TemplateTitleMixin, ListView):
    model = DigitalProduct
    template_name = "products/product_list.html"
    title = "Productos Digitales"


class ProductListView(TemplateTitleMixin, ListView):
    
    # app_label = "products"
    # model = Product
    # view_name = list
    # template_name = <app_name>/<model>_<view_name>.html>
    # template_name = products/product_list.html
    model = Product
    title = "Productos Fisicos"

     
class ProductDetailView(DetailView):
    model = Product
    
class ProtectedProductDetailView(LoginRequiredMixin, DetailView):
    model = Product

# urls.py

In [ ]:
#urls.py agrege este url

from django.contrib import admin
from django.urls import path, include
from django.views.generic import TemplateView, RedirectView

from .views import ProtectedListView

from products.views import (
    ProductListView, 
    ProductDetailView, 
    DigitalProductListView, 
    ProductIDRedirectView, 
    ProductRedirectView, 
    ProtectedProductDetailView, 
    ProtectedProductCreateView, 
    ProtectedProductUpdateView, 
    ProtectedProductDeleteView, 
)

urlpatterns = [
    path("admin/", admin.site.urls),
    

    path("about-us/", RedirectView.as_view(url="/products/about/")),
    path("about/", TemplateView.as_view(template_name="about.html")),
    path("team/", TemplateView.as_view(template_name="team.html")), 
    path("products/", ProductListView.as_view()), 
    path("digital-products/", DigitalProductListView.as_view()),
    path("products/<int:pk>/", ProductDetailView.as_view()),
    path("products/<slug:slug>/", ProductDetailView.as_view()), 
    path("p/<int:pk>/", ProductIDRedirectView.as_view()), 
    
    path("p/<slug:slug>/", ProductRedirectView.as_view()), 
    
   # path("my-products/<slug:slug>/", ProtectedProductDetailView.as_view()), 
    path("my-products/create/", ProtectedProductCreateView.as_view()), 
    path("my-products/<slug:slug>/", ProtectedProductUpdateView.as_view()), 
    path("my-products/<slug:slug>/delete/", ProtectedProductDeleteView.as_view()), 
    
    path("my-products/", ProtectedListView.as_view(),name="my-products" ), #<----- esta parte agregada
 
] 




# Templates

In [ ]:
# templates/products/my-products.html cree ese template 

<h1>Mis Productos</h1>

<ul>
    {% for product in products %}
        <li>
            {{ product.title }}-
            <a href="{{ product.get_edit_url }}">Editar</a>
        </li>
    {% empty %}
        <li>No tienes productos.</li>
    {% endfor %}
</ul>